# Phase 1: Subjective Busyness Prediction Model Comparison

Evaluates how much manager-rated floor busyness (`busyness`: 1–5) is explained by hierarchical blocks of patient care volume (`M1`), workload inequality (`M2`), and ward clinical events (`M3`) on weekdays.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().resolve().parent))
from src.config_loader import load_config
from src.stats_compat import smf

cfg = load_config()
PATHS = cfg["resolved_paths"]
ANALYSIS_CFG = cfg["analysis"]
FLOORS = [str(f) for f in ANALYSIS_CFG["floors"]]
COLORS_FLOOR = {str(k): str(v) for k, v in ANALYSIS_CFG["floor_colors"].items()}
HOSPITAL_VARS = list(ANALYSIS_CFG["hospital_vars"])

## 1. Data Loading & Weekday Filtering

In [ ]:
df = pd.read_csv(PATHS["floor_daily_staffing_summary"])
df["floor"] = df["floor"].astype(int).astype(str)
df["shift_date"] = pd.to_datetime(df["shift_date"])

df["is_weekend"] = ((df.get("dow_5", 0) == 1) | (df.get("dow_6", 0) == 1)).astype(int)
df = df[df["is_weekend"] == 0].copy()

df["care_per_staff_hour"] = df["care_level_sum"] / df["total_staff_hours"]

all_cols = ["busyness", "care_per_staff_hour", "care_std_per_staff", "floor"] + HOSPITAL_VARS
df_model = df[all_cols].dropna()

print(f"Weekday records: {len(df_model)}")
print(f"Floors: {sorted(df_model.floor.unique())}")

## 2. Per-Floor Correlation Matrix

In [ ]:
corr_cols = ["busyness", "care_per_staff_hour", "care_std_per_staff"] + HOSPITAL_VARS
short_names = ["busyness", "care/staff_hr", "care_std", "inpatients", "admitted", "nursing_proc"]

fig, axes = plt.subplots(1, len(FLOORS), figsize=(7 * len(FLOORS), 6))
axes = np.atleast_1d(axes)
for ax, fl in zip(axes, FLOORS):
    sub = df_model[df_model["floor"] == fl][corr_cols]
    corr = sub.corr()
    corr.index = short_names
    corr.columns = short_names
    sns.heatmap(
        corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
        vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax, annot_kws={"size": 9}
    )
    ax.set_title(f"Floor {fl} (n={len(sub)})")
plt.suptitle("Per-Floor Correlation Matrix (Weekdays)", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 3. Hierarchical Regression Models (M1 / M2 / M3)

- **M1**: `busyness ~ care_per_staff_hour` (workload volume)
- **M2**: `busyness ~ care_per_staff_hour + care_std_per_staff` (+ workload disparity)
- **M3**: `busyness ~ care_per_staff_hour + care_std_per_staff + hospital_vars` (+ ward clinical events)

In [ ]:
# --- [REDACTED: PROPRIETARY LOGIC] ---
# Conceptually, this step defines a domain-curated 3-tier feature block taxonomy
# (acuity volume, workload inequality, and parsimonious clinical event covariates)
# tailored to prevent small-sample overfitting.
# Below is a simplified baseline regression specification returning identical table structures.
model_defs = {
    "M1: care/staff_hr": "busyness ~ care_per_staff_hour",
    "M2: + care_std": "busyness ~ care_per_staff_hour + care_std_per_staff",
    "M3: + hospital": "busyness ~ care_per_staff_hour + num_inpatients",
}

floor_results = {}
for fl in FLOORS:
    sub = df_model[df_model["floor"] == fl]
    floor_results[fl] = {
        name: smf.ols(formula, sub).fit()
        for name, formula in model_defs.items()
    }
    floor_results[fl]["n"] = len(sub)

print("Per-Floor Hierarchical Model Comparison (M1/M2/M3, weekdays):")
print(f"{'Floor':<7} {'Model':<25} {'R2':>8} {'Adj_R2':>8} {'RMSE':>8} {'n':>5}")
print("-" * 60)
for fl, rd in floor_results.items():
    for name in model_defs:
        res = rd[name]
        print(f"{fl:<7} {name:<25} {res.rsquared:>8.4f} {res.rsquared_adj:>8.4f} {np.sqrt(res.mse_resid):>8.4f} {rd['n']:>5}")
    print()

In [ ]:
model_names = list(model_defs.keys())
colors_step = ["#fee08b", "#fc8d59", "#d73027"]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
x = np.arange(len(FLOORS))
width = 0.25
for i, (mname, color) in enumerate(zip(model_names, colors_step)):
    vals = [floor_results[fl][mname].rsquared_adj for fl in FLOORS]
    bars = axes[0].bar(x + i * width, vals, width, label=mname, color=color, edgecolor="black")
    for bar, val in zip(bars, vals):
        axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{val:.3f}", ha="center", va="bottom", fontsize=9)
axes[0].set_xticks(x + width)
axes[0].set_xticklabels([f"Floor {fl}" for fl in FLOORS])
axes[0].set_ylabel("Adjusted R²")
axes[0].set_title("Hierarchical Model (M1/M2/M3): Adj R² by Floor")
axes[0].legend(fontsize=9)
axes[0].axhline(0, color="gray", linestyle="--", linewidth=0.5)

for fl in FLOORS:
    sub = df_model[df_model["floor"] == fl]
    axes[1].scatter(
        sub["care_std_per_staff"],
        sub["busyness"] + np.random.uniform(-0.15, 0.15, len(sub)),
        alpha=0.5, s=25, color=COLORS_FLOOR.get(fl, "steelblue"), label=f"Floor {fl}"
    )
axes[1].set_xlabel("Care Std per Staff (Workload Disparity)")
axes[1].set_ylabel("Busyness")
axes[1].set_title("Workload Disparity vs Busyness")
axes[1].legend()

plt.tight_layout()
plt.show()